#Task 1: Load and Inspect the Data

In [10]:
import pandas as pd
import numpy as np

data = {
    'transaction_id': range(1, 21),
    'date': pd.date_range('2024-10-01', periods=20, freq='D'),
    'region': ['North', 'South', 'East', 'West', None, 'North', 'South', None, 'East', 'West',
               'North', 'South', 'East', 'West', 'North', None, 'East', 'West', 'North', 'South'],
    'product_category': ['Electronics', 'Clothing', None, 'Books', 'Electronics', 'Home',
                         'Clothing', 'Books', 'Electronics', None, 'Home', 'Clothing',
                         'Books', 'Electronics', 'Home', 'Clothing', 'Books', 'Electronics',
                         'Home', 'Clothing'],
    'sales_amount': [1200, 450, 890, None, 1500, 670, None, 340, 2100, 780,
                     560, None, 420, 1800, 920, 510, 380, None, 1100, 640],
    'quantity': [2, 5, 3, 1, None, 4, 2, 3, 1, 5,
                 3, 2, None, 1, 4, 3, 2, 1, None, 4],
    'customer_age': [25, 34, None, 45, 29, None, 38, 52, 27, 41,
                     33, None, 48, 26, 35, 42, None, 31, 39, 44],
    'payment_method': ['Credit Card', 'UPI', 'Cash', 'Debit Card', 'Credit Card',
                       'UPI', 'Cash', None, 'Credit Card', 'UPI', 'Debit Card',
                       'Cash', 'Credit Card', None, 'UPI', 'Cash', 'Debit Card',
                       'Credit Card', 'UPI', None]
}


# Create a DataFrame from the variable `data`
df = pd.DataFrame(data)

# Display a summary of the DataFrame:
df.info()

# Count how many missing (NaN) values each column has
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    20 non-null     int64         
 1   date              20 non-null     datetime64[ns]
 2   region            17 non-null     object        
 3   product_category  18 non-null     object        
 4   sales_amount      16 non-null     float64       
 5   quantity          17 non-null     float64       
 6   customer_age      16 non-null     float64       
 7   payment_method    17 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 1.4+ KB


,0
transaction_id,0
date,0
region,3
product_category,2
sales_amount,4
quantity,3
customer_age,4
payment_method,3


#Task 2: Handle Missing Values

In [12]:

# Fill missing values in the 'region' column with the most frequent (mode) value.
df["region"] = df["region"].fillna(df["region"].mode()[0])

# Fill missing values in 'product_category' using that column’s mode (most common category).
df["product_category"] = df["product_category"].fillna(df["product_category"].mode()[0])

# Fill missing values in 'sales_amount' with the median value of the column.
df["sales_amount"] = df["sales_amount"].fillna(df["sales_amount"].median())

# Fill missing values in 'quantity' using forward-fill (ffill)
df["quantity"] = df["quantity"].fillna(method="ffill")

# Calculate the average customer age, rounded to the nearest integer.
avg_customer_age = int(round(df["customer_age"].mean(), 0))

# Fill missing ages with the computed average age.
df["customer_age"] = df["customer_age"].fillna(avg_customer_age)

# Drop any rows where 'payment_method' is missing.
df = df.dropna(subset=["payment_method"])

# Display the count of remaining missing values in each column.
df.isna().sum()

/tmp/ipython-input-410/1738576542.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["quantity"] = df["quantity"].fillna(method="ffill")


,0
transaction_id,0
date,0
region,0
product_category,0
sales_amount,0
quantity,0
customer_age,0
payment_method,0


#Task 3: GroupBy Analysis

In [8]:

# Group the data by 'region' and calculate the total (sum) of sales_amount for each region.
print(df.groupby("region")["sales_amount"].sum())

print("\n")  # Print a blank line for readability

# Group the data by 'product_category' and calculate the average (mean) sales amount per category.
print(df.groupby("product_category")["sales_amount"].mean())

print("\n")  # Blank line for spacing

# Group by both 'region' and 'product_category' and compute:
#   - total sales_amount
#   - total quantity
# reset_index() converts the grouped index back to normal columns for cleaner output.
print(
    df.groupby(["region", "product_category"])
      .agg({"sales_amount": "sum", "quantity": "sum"})
      .reset_index()
)

print("\n")  # Blank line for spacing

# Show the top 3 regions with the highest total sales,
# by sorting the region-wise sales sum in descending order.
print(df.groupby("region")["sales_amount"].sum().sort_values(ascending=False).head(3))


region
East     3790.0
North    6460.0
South    1900.0
West     2230.0
Name: sales_amount, dtype: float64


product_category
Books           508.333333
Clothing        680.000000
Electronics    1381.250000
Home            812.500000
Name: sales_amount, dtype: float64


  region product_category  sales_amount  quantity
0   East            Books         800.0       4.0
1   East         Clothing         890.0       3.0
2   East      Electronics        2100.0       1.0
3  North         Clothing         510.0       3.0
4  North      Electronics        2700.0       3.0
5  North             Home        3250.0      12.0
6  South         Clothing        1900.0       9.0
7   West            Books         725.0       1.0
8   West         Clothing         780.0       5.0
9   West      Electronics         725.0       1.0


region
North    6460.0
East     3790.0
West     2230.0
Name: sales_amount, dtype: float64


#Task 4: Custom Aggregation

In [14]:

# Define a custom aggregation function that calculates the spread of values:
def sales_range(value):
    return value.max() - value.min()

# 1) For each region, compute three metrics on 'sales_amount':
#    - 'max':   the highest sales amount in that region
#    - 'min':   the lowest sales amount in that region
#    - sales_range: the custom spread (max - min) for that region
print(df.groupby("region")["sales_amount"].agg(["max", "min", sales_range]))

print("\n")  # Blank line for readability

# 2) A multi-aggregation across multiple columns:
#    - For 'sales_amount': compute total (sum), average (mean), and maximum (max)
#    - For 'quantity': compute total (sum) and smallest value (min)
df.groupby("region").agg({
    "sales_amount": ["sum", "mean", "max"],
    "quantity":     ["sum", "min"]
})


           max    min  sales_range
region                            
East    2100.0  380.0       1720.0
North   1500.0  510.0        990.0
South    725.0  450.0        275.0
West     780.0  725.0         55.0




sales_amount                     quantity     
                sum        mean     max      sum  min
region                                               
East         3790.0  947.500000  2100.0      8.0  1.0
North        6460.0  922.857143  1500.0     18.0  1.0
South        1900.0  633.333333   725.0      9.0  2.0
West         2230.0  743.333333   780.0      7.0  1.0